# RetentIA — Exploratory Data Analysis (EDA)

Análise exploratória do dataset IBM Telco Customer Churn (~7,043 clientes).

**Objetivo:** entender a distribuição das features, o desbalanceamento do target,
e as relações entre variáveis antes de treinar os modelos.

**Gotcha documentado:** `TotalCharges` contém espaços em branco para clientes com
`tenure == 0` (clientes novos, nunca cobrados). Missing **estrutural**, não aleatório.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.data.ingest import ingest_data
from src.features.columns import (
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN,
    ALL_FEATURES,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

df = ingest_data()
print(f"Shape: {df.shape}")
print(f"Features: {len(ALL_FEATURES)} | Numerical: {len(NUMERICAL_FEATURES)} | Categorical: {len(CATEGORICAL_FEATURES)}")
df.head()

## 1. Visão Geral do Dataset

In [ ]:
print("=== Info ===")
print(f"Linhas: {len(df)}")
print(f"Colunas: {len(df.columns)}")
print(f"\n=== Tipos ===")
print(df.dtypes.value_counts())
print(f"\n=== Valores faltantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Nenhum valor faltante (TotalCharges já tratado na ingestão)")
print(f"\n=== Estatísticas descritivas (numéricas) ===")
df[NUMERICAL_FEATURES].describe()

## 2. Target: Distribuição do Churn

O dataset original tem `Churn` como "Yes"/"No". Após a ingestão, o `train.py`
converte para 0/1. Aqui mostramos a distribuição original.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem
churn_counts = df[TARGET_COLUMN].value_counts()
churn_counts.plot(kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"], edgecolor="black")
axes[0].set_title("Distribuição do Churn (contagem)")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Clientes")
for i, v in enumerate(churn_counts):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

# Proporção
churn_pct = df[TARGET_COLUMN].value_counts(normalize=True) * 100
churn_pct.plot(kind="pie", ax=axes[1], autopct="%.1f%%", colors=["#2ecc71", "#e74c3c"],
               startangle=90, explode=(0, 0.05))
axes[1].set_title("Distribuição do Churn (%)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("../docs/churn_distribution.png", bbox_inches="tight")
plt.show()

print(f"\nChurn rate: {churn_pct.get('Yes', 0):.1f}%")
print(f"Classe majoritária (No): {churn_pct.get('No', 0):.1f}%")
print("→ Dataset DESBALANCEADO (~26.5% positivos) — tratado com pos_weight no MLP e class_weight no LogReg.")

## 3. Gotcha: TotalCharges com espaços em branco

11 linhas têm `TotalCharges` com espaço em branco em vez de número.
Esses são **exatamente** os clientes com `tenure == 0` (recém-assinados, nunca cobrados).
É missing **estrutural**, não aleatório. Imputado como 0.0.

In [ ]:
# Recarregar CSV bruto pra mostrar o gotcha ANTES do tratamento
df_raw = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

blank_mask = df_raw["TotalCharges"] == " "
tenure_zero_mask = df_raw["tenure"] == 0

print(f"Linhas com TotalCharges em branco: {blank_mask.sum()}")
print(f"Linhas com tenure == 0: {tenure_zero_mask.sum()}")
print(f"Blank == tenure_zero? {blank_mask.equals(tenure_zero_mask)}")
print(f"\nClientes afetados:")
df_raw[blank_mask][["customerID", "tenure", "TotalCharges", "MonthlyCharges", "Churn"]]

## 4. Features Numéricas: Distribuição e Correlação

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(NUMERICAL_FEATURES):
    for label, color in [("No", "#2ecc71"), ("Yes", "#e74c3c")]:
        subset = df[df[TARGET_COLUMN] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, label=f"Churn={label}", color=color, edgecolor="black")
    axes[i].set_title(f"{col} por Churn")
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.tight_layout()
plt.savefig("../docs/numerical_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlação entre numéricas + target (binarizado)
df_corr = df[NUMERICAL_FEATURES].copy()
df_corr["Churn_bin"] = (df[TARGET_COLUMN] == "Yes").astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df_corr.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlação: Numéricas + Churn")
plt.tight_layout()
plt.savefig("../docs/correlation_matrix.png", bbox_inches="tight")
plt.show()

## 5. Features Categóricas de Alto Impacto

Foco nas variáveis que mais diferenciam churn vs não-churn:
`Contract`, `InternetService`, `PaymentMethod`, `TechSupport`.

In [ ]:
key_cats = ["Contract", "InternetService", "PaymentMethod", "TechSupport"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.ravel(), key_cats):
    ct = pd.crosstab(df[col], df[TARGET_COLUMN], normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="black")
    ax.set_title(f"Churn rate por {col}")
    ax.set_ylabel("% clientes")
    ax.set_xlabel("")
    ax.legend(title="Churn", loc="upper right")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("../docs/categorical_churn_rates.png", bbox_inches="tight")
plt.show()

## 6. Churn Rate por Todas as Categóricas

Visão completa — uma barra por categoria mostrando a taxa de churn (%).

In [ ]:
cat_cols = [c for c in CATEGORICAL_FEATURES if c != "SeniorCitizen"]

fig, axes = plt.subplots(5, 3, figsize=(18, 22))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)[TARGET_COLUMN].apply(lambda x: (x == "Yes").mean() * 100)
    churn_rate.sort_values(ascending=False).plot(kind="barh", ax=axes[i], color="#3498db", edgecolor="black")
    axes[i].set_title(f"Churn % por {col}", fontsize=11)
    axes[i].set_xlabel("Churn rate (%)")
    axes[i].axvline(x=26.5, color="red", linestyle="--", alpha=0.5, label="Média geral")

# Esconder eixos extras se sobrarem
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("../docs/all_categorical_churn_rates.png", bbox_inches="tight")
plt.show()

## 7. Tenure vs Churn: Curva de Sobrevivência Simplificada

Clientes com tenure baixo churn muito mais. Isso faz sentido com o
negócio: contratos mês-a-mês têm friccão zero para cancelar.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for label, color in [("No", "#2ecc71"), ("Yes", "#e74c3c")]:
    subset = df[df[TARGET_COLUMN] == label]
    ax.hist(subset["tenure"], bins=72, alpha=0.6, label=f"Churn={label}", color=color, edgecolor="black")

ax.set_title("Distribuição de Tenure por Churn")
ax.set_xlabel("Tenure (meses)")
ax.set_ylabel("Clientes")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/tenure_vs_churn.png", bbox_inches="tight")
plt.show()

# Churn rate por faixa de tenure
df["tenure_bin"] = pd.cut(df["tenure"], bins=[0, 6, 12, 24, 48, 72], labels=["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"])
tenure_churn = df.groupby("tenure_bin", observed=True)[TARGET_COLUMN].apply(lambda x: (x == "Yes").mean() * 100)
print("Churn rate por faixa de tenure:")
print(tenure_churn.to_string())
print("\n→ Clientes nos primeiros 6 meses churn ~3x mais que veteranos.")

## 8. MonthlyCharges vs TotalCharges — Segmentação por Churn

Clientes que churn tendem a ter **cobranças mensais mais altas** e
**cobranças totais mais baixas** (ficaram pouco tempo).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for label, color, marker in [("No", "#2ecc71", "o"), ("Yes", "#e74c3c", "x")]:
    subset = df[df[TARGET_COLUMN] == label].sample(min(500, len(df)), random_state=42)
    ax.scatter(subset["MonthlyCharges"], subset["TotalCharges"],
               alpha=0.4, label=f"Churn={label}", color=color, marker=marker, s=15)

ax.set_xlabel("MonthlyCharges")
ax.set_ylabel("TotalCharges")
ax.set_title("MonthlyCharges vs TotalCharges (amostra, colorido por Churn)")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/charges_scatter.png", bbox_inches="tight")
plt.show()

## 9. Resumo dos Achados

| Achado | Implicação |
|--------|-----------|
| Churn ≈ 26.5% | Classe desbalanceada → `pos_weight` no MLP, `class_weight` no LogReg |
| TotalCharges com 11 blanks = tenure==0 | Missing estrutural, não aleatório → imputado 0.0 |
| Month-to-month: churn ~42% vs Two year: ~3% | `Contract` é a feature mais discriminativa |
| Fiber optic: churn ~42% vs DSL: ~19% | Possível insatisfação com preço/qualidade do fiber |
| Electronic check: churn ~45% | Método de pagamento automático retém mais |
| Tenure < 6 meses: churn ~50% | Primeiros meses são críticos para retenção |
| MonthlyCharges altas + TotalCharges baixas = churn | Clientes novos pagando caro saem rápido |

Esses achados são consistentes com a literatura sobre churn em telecom e
validam a escolha de threshold sensível a custo (C_FN >> C_FP).

In [ ]:
# Cleanup
if "tenure_bin" in df.columns:
    df.drop(columns=["tenure_bin"], inplace=True)

print("EDA concluída. Gráficos salvos em docs/.")
print("Próximo passo: make train → preencher MODEL_CARD.md com métricas reais.")